In [2]:
from collections import defaultdict, deque
import re

# Parse states from the first part of a machine line
def parse_states(header):
    tokens = re.findall(r'q\d+f?', header.strip())

    states = []
    final_states = set()

    for token in tokens:
        if token.endswith("f"):
            state = token[:-1]
            states.append(state)
            final_states.add(state)
        else:
            states.append(token)

    start_state = states[0]
    return states, start_state, final_states

# Parse NFA
def parse_nfa(line):
    parts = [p.strip() for p in line.strip().split(",") if p.strip()]

    header = parts[0]
    states, start_state, final_states = parse_states(header)

    transitions = defaultdict(list)

    for part in parts[1:]:
        if "->" not in part:
            continue

        left, right = part.split("->")
        right = right.strip()

        if "-" not in left:
            continue

        state, symbols = left.split("-", 1)
        state = state.strip()

        for symbol in symbols.split("|"):
            symbol = symbol.strip()
            transitions[(state, symbol)].append(right)

    return {
        "states": states,
        "start": start_state,
        "final": final_states,
        "transitions": transitions
    }

# Parse NPDA
def parse_npda(line):
    parts = [p.strip() for p in line.strip().split(",") if p.strip()]

    header = parts[0]
    states, start_state, final_states = parse_states(header)

    transitions = defaultdict(list)

    for part in parts[1:]:
        if "->" not in part:
            continue

        left, right = part.split("->")
        next_state = right.strip()

        pieces = left.split("-")
        if len(pieces) < 4:
            continue

        current_state = pieces[0].strip()
        rest = "-".join(pieces[1:])
        options = rest.split("|")

        for option in options:
            items = option.split("-")
            if len(items) != 3:
                continue

            input_symbol = items[0].strip()
            pop_symbol = items[1].strip()
            push_symbol = items[2].strip()

            transitions[current_state].append(
                (input_symbol, pop_symbol, push_symbol, next_state)
            )

    return {
        "states": states,
        "start": start_state,
        "final": final_states,
        "transitions": transitions
    }

# Build intersection NPDA
# New state = (NFA state, NPDA state)
#
# If NPDA reads a or b:
#   NFA must also move on same symbol
#
# If NPDA reads empty:
#   NFA stays in same state
#
# Final state:
#   both NFA state and NPDA state must be final
def build_intersection_npda(nfa, npda):
    new_states = []
    new_final = set()
    new_transitions = defaultdict(list)

    for nfa_state in nfa["states"]:
        for npda_state in npda["states"]:
            pair = (nfa_state, npda_state)
            new_states.append(pair)

            if nfa_state in nfa["final"] and npda_state in npda["final"]:
                new_final.add(pair)

    start_pair = (nfa["start"], npda["start"])

    for nfa_state in nfa["states"]:
        for npda_state in npda["states"]:
            current_pair = (nfa_state, npda_state)

            for inp, pop_sym, push_sym, npda_next in npda["transitions"].get(npda_state, []):

                # epsilon move in NPDA
                if inp == "empty":
                    next_pair = (nfa_state, npda_next)
                    new_transitions[current_pair].append(
                        (inp, pop_sym, push_sym, next_pair)
                    )

                # normal move on a or b
                else:
                    for nfa_next in nfa["transitions"].get((nfa_state, inp), []):
                        next_pair = (nfa_next, npda_next)
                        new_transitions[current_pair].append(
                            (inp, pop_sym, push_sym, next_pair)
                        )

    return {
        "states": new_states,
        "start": start_pair,
        "final": new_final,
        "transitions": new_transitions
    }

# Convert pair state to string
def pair_to_string(pair):
    return pair[0] + pair[1]

# Print NPDA in one-line format
def format_npda(npda):
    header = ""

    for state in npda["states"]:
        name = pair_to_string(state)
        if state in npda["final"]:
            header += name + "f"
        else:
            header += name

    parts = [header]
    grouped = defaultdict(list)

    for source in npda["states"]:
        for inp, pop_sym, push_sym, dest in npda["transitions"].get(source, []):
            grouped[(source, dest)].append(f"{inp}-{pop_sym}-{push_sym}")

    for source in npda["states"]:
        for dest in npda["states"]:
            if (source, dest) in grouped:
                left = pair_to_string(source)
                right = pair_to_string(dest)
                middle = "|".join(grouped[(source, dest)])
                parts.append(f"{left}-{middle}->{right}")

    return ",".join(parts)

# Run NPDA on a string
# Stack starts with z
# Accept only if:
# input finished and current state is final
def run_npda(npda, test_string):
    start_state = npda["start"]
    start_stack = ("z",)

    queue = deque()
    queue.append((start_state, 0, start_stack))

    visited = set()
    max_steps = 10000
    steps = 0

    while queue and steps < max_steps:
        steps += 1
        state, pos, stack = queue.popleft()

        config = (state, pos, stack)
        if config in visited:
            continue
        visited.add(config)

        if pos == len(test_string) and state in npda["final"]:
            return "accept"

        for inp, pop_sym, push_sym, next_state in npda["transitions"].get(state, []):
            new_pos = pos
            new_stack = list(stack)

            # input check
            if inp != "empty":
                if pos >= len(test_string):
                    continue
                if test_string[pos] != inp:
                    continue
                new_pos += 1

            # pop check
            if pop_sym != "empty":
                if not new_stack:
                    continue
                if new_stack[-1] != pop_sym:
                    continue
                new_stack.pop()

            # push
            if push_sym != "empty":
                for ch in reversed(push_sym):
                    new_stack.append(ch)

            queue.append((next_state, new_pos, tuple(new_stack)))

    return "reject"

# Helper to run one test case
def test_case(case_number, nfa_line, npda_line, test_strings):
    print(f"\nTest Case {case_number}")
    print("-" * 50)

    nfa = parse_nfa(nfa_line)
    npda = parse_npda(npda_line)

    intersection = build_intersection_npda(nfa, npda)

    print("Output NPDA:")
    print(format_npda(intersection))
    print()

    for s in test_strings:
        print(f"{s} -> {run_npda(intersection, s)}")

# MAIN
if __name__ == "__main__":

    # Test Case 1
    nfa1 = "q1q2q3f,q1-a->q1,q1-b->q2,q2-a->q1,q2-b->q3,q3-a|b->q3"
    npda1 = "q1q2f,q1-a-z-az|a-a-empty|b-z-z|b-a-a->q1,q1-empty-z-z->q2"
    strings1 = ["aabbaa", "aaaa"]

    # Test Case 2
    nfa2 = "q1fq2fq3fq4,q1-a->q1,q1-b->q2,q2-a->q2,q2-b->q3,q3-a->q3,q3-b->q4,q4-a|b->q4"
    npda2 = "q1q2q3q4fq5,q1-empty-z-az->q2,q2-empty-a-aa->q3,q3-empty-a-aa->q4,q4-a-a-empty|b-z-z|b-a-a->q4"
    strings2 = ["baaba", "baaaab"]

    # Test Case 3
    nfa3 = "q1fq2fq3fq4fq5q6f,q1-a->q2,q1-b->q1,q2-a->q3,q2-b->q1,q3-a->q4,q3-b->q1,q4-a->q5,q4-b->q1,q5-a->q6,q5-b->q1,q6-a->q6,q6-b->q1"
    npda3 = "q1q2q3f,q1-a-z-z->q1,q1-b-z-z->q2,q2-a-az-az|a-a-ab|b-z-z|b-a-empty->q2,q2-empty-z-z->q3"
    strings3 = ["baaa", "aabaaab"]

    # Test Case 4
    # Adjusted to use dashes in transitions
    nfa4 = "q1q2fq3q4q5f,q1-empty->q2,q1-empty->q4,q2-a->q2,q2-b->q3,q3-a->q3,q3-b->q2,q4-b->q4,q4-a->q5,q5-b->q5,q5-a->q4"
    npda4 = "q1q2q3f,q1-a-z-z->q1,q1-b-z-z->q2,q2-b-z-bz|b-b-empty|a-z-z|a-b-b->q2,q2-empty-z-z->q3"
    strings4 = ["bbaaa", "baaabbb"]

    # Test Case 5
    # I also fixed the likely state-header typo q1fg2q3f -> q1fq2q3f
    nfa5 = "q1fq2q3f,q1-a|b->q2,q2-a|b->q3,q3-a|b->q1"
    npda5 = "q1q2q3f,q1-empty-z-az->q2,q2-a-a-empty|b-a-empty|a-z-bz|b-z-bz|a-b-a|b-b-a|empty-z-z->q3"
    strings5 = []

    test_case(1, nfa1, npda1, strings1)
    test_case(2, nfa2, npda2, strings2)
    test_case(3, nfa3, npda3, strings3)
    test_case(4, nfa4, npda4, strings4)

    if strings5:
        test_case(5, nfa5, npda5, strings5)


Test Case 1
--------------------------------------------------
Output NPDA:
q1q1q1q2q2q1q2q2q3q1q3q2f,q1q1-a-z-az|a-a-empty->q1q1,q1q1-empty-z-z->q1q2,q1q1-b-z-z|b-a-a->q2q1,q2q1-a-z-az|a-a-empty->q1q1,q2q1-empty-z-z->q2q2,q2q1-b-z-z|b-a-a->q3q1,q3q1-a-z-az|a-a-empty|b-z-z|b-a-a->q3q1,q3q1-empty-z-z->q3q2

aabbaa -> accept
aaaa -> reject

Test Case 2
--------------------------------------------------
Output NPDA:
q1q1q1q2q1q3q1q4fq1q5q2q1q2q2q2q3q2q4fq2q5q3q1q3q2q3q3q3q4fq3q5q4q1q4q2q4q3q4q4q4q5,q1q1-empty-z-az->q1q2,q1q2-empty-a-aa->q1q3,q1q3-empty-a-aa->q1q4,q1q4-a-a-empty->q1q4,q1q4-b-z-z|b-a-a->q2q4,q2q1-empty-z-az->q2q2,q2q2-empty-a-aa->q2q3,q2q3-empty-a-aa->q2q4,q2q4-a-a-empty->q2q4,q2q4-b-z-z|b-a-a->q3q4,q3q1-empty-z-az->q3q2,q3q2-empty-a-aa->q3q3,q3q3-empty-a-aa->q3q4,q3q4-a-a-empty->q3q4,q3q4-b-z-z|b-a-a->q4q4,q4q1-empty-z-az->q4q2,q4q2-empty-a-aa->q4q3,q4q3-empty-a-aa->q4q4,q4q4-a-a-empty|b-z-z|b-a-a->q4q4

baaba -> accept
baaaab -> reject

Test Case 3
----------------------